# Fast and Slow: Colab entry point

**Can a System One decision model (Kev-4B, an open JEV-style model) take over a planner's decidable calls, and how does it learn compared with classic RL?**
The spec, with its feature map and the reasoning behind each decision, is in `docs/spec.md`.

This notebook is deliberately thin: every cell calls one `fns` command from the tested `fast_and_slow` package.

**Runs on an A100** (Runtime > Change runtime type > A100 GPU; H100s are rarely available). Don't pick a TPU: Kev needs an NVIDIA GPU.

**Built for a compute-unit budget (about 76 CU, spent in sessions of about 15).**
- Only Kev runs here, on the GPU. PPO needs only a CPU, so it runs on your laptop for free (section 8).
- `kev-check` measures Kev's real speed on whichever GPU you got, and `plan` prices every stage in compute units *before* you spend them.
- Kev RL stops at a checkpoint when the session's share runs out (`SESSION_CU`), and the next session carries on.

**Built for disconnects.** Colab Pro sessions end after about 24 hours at most, often sooner, and drop when idle.
- Finished stages are skipped.
- Kev RL resumes exactly from its last checkpoint (every 3 updates, saved to Google Drive).
- After a disconnect: run sections 1–3, then the cell you were in.

## 1. Settings

In [ ]:
REPO = "https://github.com/carlosdonado10/fast-and-slow.git"   # private repo: add a GITHUB_TOKEN secret (key icon, left sidebar)
BRANCH = "main"
PROFILE = "full"                                                # "smoke" runs everything in minutes with a fake Kev
RUNS = "/content/drive/MyDrive/fast-and-slow/runs"              # survives disconnects; your laptop reads it too
CU_PER_HOUR = 11.8      # A100 (the default runtime), approximate: check Runtime > View resources and correct it if needed
BUDGET_CU = 76          # what's left on your account
SESSION_CU = 15         # what you'll spend per sitting; every session first pays ~2 CU of setup (model download)
RUN_LABEL = "A100 study"   # how this run is named in the report's run picker
MAX_MINUTES = int(SESSION_CU / CU_PER_HOUR * 60) - 15   # Kev RL pauses at a checkpoint after this; the next session continues

## 2. Setup: Drive, code, environment
The first session installs the environment and saves a copy to Drive; later sessions unpack it in about a minute.


In [ ]:
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(RUNS, exist_ok=True)

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = REPO.replace("https://", f"https://{token}@") if token else REPO

# /content is the runtime's own disk: wiped with every new session, so the code is cloned once per session
# (a shallow clone, a few seconds) and pulled when the cell runs again. Only RUNS lives on Drive.
if not os.path.exists("/content/fast-and-slow"):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", url, "/content/fast-and-slow"], check=True)
else:
    subprocess.run(["git", "-C", "/content/fast-and-slow", "pull", "--ff-only"], check=True)
os.chdir("/content/fast-and-slow/backend")

In [ ]:
import hashlib, json, pathlib, platform, shutil, time
# The environment (PyTorch, CUDA libraries, Kev; several GB) is installed once, then saved to Drive as ONE archive.
# A new session unpacks it instead of reinstalling. The archive is named after uv.lock, so a dependency change makes
# a fresh one (and the old one is deleted). Set CACHE_ENV = False to skip it if Drive space is tight (it takes ~6 GB).
CACHE_ENV = True

t0 = time.monotonic()
backend = pathlib.Path("/content/fast-and-slow/backend")
venv = backend / ".venv"
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
os.environ["UV_PYTHON_PREFERENCE"] = "only-system"   # build on Colab's own Python, so the saved venv works next session
if not shutil.which("uv"):
    subprocess.run("curl -LsSf https://astral.sh/uv/install.sh | sh", shell=True, check=True, capture_output=True)

key = hashlib.sha256((backend / "uv.lock").read_bytes() + platform.python_version().encode()).hexdigest()[:12]
cache = pathlib.Path(RUNS).parent / "cache"
archive = cache / f"env-{key}.tar"

if venv.exists():
    print("environment: already installed in this session")
elif CACHE_ENV and archive.exists():
    print(f"environment: unpacking the saved copy from Drive ({archive.stat().st_size / 2**30:.1f} GB)...")
    subprocess.run(["tar", "-xf", str(archive), "-C", str(backend)], check=True)
    took = archive.with_suffix(".json")
    if took.exists():  # is this actually faster than installing? the first session recorded how long that took
        install_s = json.loads(took.read_text())["install_seconds"]
        unpack_s = time.monotonic() - t0
        print(f"environment: unpacked in {unpack_s:.0f}s; installing took {install_s:.0f}s "
              f"({'faster' if unpack_s < install_s else 'SLOWER: set CACHE_ENV = False'})")

fresh = not venv.exists()
if fresh:
    print("environment: installing (first session, or the dependencies changed); a few minutes...")
t_install = time.monotonic()
r = subprocess.run(["uv", "sync", "--extra", "gpu", "--frozen"], cwd=backend, capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise RuntimeError("uv sync failed")

install_seconds = time.monotonic() - t_install
if fresh:
    print(f"environment: installed in {install_seconds:.0f}s")
if fresh and CACHE_ENV:
    print("environment: saving a copy to Drive for the next sessions...")
    cache.mkdir(parents=True, exist_ok=True)
    tmp = archive.with_suffix(".tmp")
    subprocess.run(["tar", "-cf", str(tmp), "-C", str(backend), ".venv"], check=True)
    tmp.rename(archive)  # only a complete archive ever gets the real name
    archive.with_suffix(".json").write_text(json.dumps({"install_seconds": install_seconds}))
    for old in [*cache.glob("env-*.tar"), *cache.glob("env-*.json")]:
        if old.stem != archive.stem:
            old.unlink()

m, s = divmod(round(time.monotonic() - t0), 60)
print(f"environment ready in {m}m{s:02d}s")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])

In [ ]:
import time
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
os.environ["PYTHONUNBUFFERED"] = "1"   # stream progress lines as they happen, not in bursts
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less memory lost to fragmentation in RL
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # optional secret: faster model downloads
except Exception:
    pass

def fns(*args):
    """Run one study stage and stream its output. Long stages print progress with an estimate of the time left."""
    cmd = ["uv", "run", "--frozen", "fns", *map(str, args), "--profile", PROFILE, "--runs", RUNS]
    started = time.monotonic()
    print(f"$ fns {' '.join(map(str, args))}", flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    try:
        for line in p.stdout:
            print(line, end="", flush=True)
            out.append(line)
    except KeyboardInterrupt:   # stopping the cell stops the stage too, instead of leaving it running unseen
        p.terminate()
        try:
            p.wait(timeout=60)
        except subprocess.TimeoutExpired:
            p.kill()
        print(f"fns {args[0]} stopped. Its last checkpoint is kept; run the cell again to continue.")
        raise
    m, s = divmod(round(time.monotonic() - started), 60)
    if p.wait() != 0:
        raise RuntimeError(f"fns {args[0]} failed after {m}m{s:02d}s")
    print(f"fns {args[0]} finished in {m}m{s:02d}s\n", flush=True)
    return "".join(out)

fns("status")

## 3. Preflight and the bill
`kev-check` loads Kev-4B, confirms gradients reach the adapter, and times real requests on this GPU (about a minute).
`plan` then prices every stage and says how many sessions of `SESSION_CU` it takes. **If it says over budget, follow its suggestion before going further.**

In [ ]:
fns("notes")       # Phi-4-mini writes the inbox notes once (about 10 min); kev-check uses them
fns("kev-check")
fns("plan", "--cu-per-hour", CU_PER_HOUR, "--budget", BUDGET_CU, "--session-cu", SESSION_CU)

## 4. References (CPU, about a minute)

In [ ]:
fns("refs")

## 5. Kev zero-shot and in-context (no training)

In [ ]:
fns("kev")

## 6. Kev + RL
Each run continues from its checkpoint. All runs together share `MAX_MINUTES` per session, then pause at a checkpoint.
Every update prints a line with how far along the run is, roughly how long is left and when it pauses.
When the session's time is used up, the cell uploads the checkpoint to Drive before telling you it's safe to disconnect.
**Don't delete the runtime while it says it's uploading.** Run this cell again (in this session or the next)
until every run says *finished*. Check progress with `fns("status")`.

In [ ]:
import yaml, pathlib
cfg = yaml.safe_load(pathlib.Path(RUNS, PROFILE, "config.json").read_text())
todo = [(h, seed) for seed in cfg["seeds"] for h in cfg["kev"]["rl_horizons"]]
session_end = time.monotonic() + MAX_MINUTES * 60
paused = False
for i, (h, seed) in enumerate(todo, 1):
    left = (session_end - time.monotonic()) / 60
    if left < 2:
        paused = True
        break
    print(f"=== RL run {i} of {len(todo)}: horizon {h}, seed {seed} ({left:.0f} min of this session's RL time left) ===")
    if "paused at a checkpoint" in fns("kev-rl", "--horizon", h, "--seed", seed, "--max-minutes", int(left)):
        paused = True
        break
fns("status")
if paused:
    # Drive uploads finish after the write returns. Flushing waits for them, so the checkpoint is really on Drive
    # before you disconnect (a deleted runtime drops anything still uploading).
    from google.colab import drive
    print("This session's RL time is used up. Making sure the checkpoint is uploaded to Drive...")
    drive.flush_and_unmount()
    print("Checkpoint saved to Drive. Safe to disconnect. Next session: sections 1 and 2, then this cell.")


In [ ]:
fns("status")

## 6b. Regime changes (H6): Kev's part, a few minutes
Kev on each new world (told about the change, and not told) and the weekly "has something changed?" alarm over
simulated years. The PPO retraining half runs on your laptop (section 8).

In [ ]:
fns("shift", "--part", "gpu")

**When everything here is done, disconnect the runtime** (Runtime > Disconnect and delete runtime) so it stops using units.

## 7. Publish the report as a run
Run this after PPO (section 8) has finished on your laptop, or now for a partial report. The report is saved on Drive
and, if the runs API is reachable, published as a new run: the report site lists every run and opens the newest.
Reports are never committed to git.

Publishing needs two Colab secrets (key icon, left sidebar): `FNS_API` (the stack's API, e.g.
`https://<your domain>/p/fast-and-slow/api`) and `UPLOAD_TOKEN` (the same value as in the stack's `.env`). If the stack
isn't online, publish later from your laptop with the command this cell prints.


In [ ]:
report = f"{RUNS}/{PROFILE}/report.json"
fns("export", "--out", report)   # saved on Drive whatever happens next

from google.colab import userdata
try:
    os.environ["FNS_API"] = userdata.get("FNS_API")
    os.environ["UPLOAD_TOKEN"] = userdata.get("UPLOAD_TOKEN")   # via the environment, so it never appears in the output
    fns("publish", "--report", report, "--label", RUN_LABEL)
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    print("Not published: add the FNS_API and UPLOAD_TOKEN secrets, or publish from your laptop with the local stack:")
    print(f'  uv run fns publish --profile {PROFILE} --report "<Drive>/fast-and-slow/runs/{PROFILE}/report.json" '
          f'--api http://localhost:8088/api --label "{RUN_LABEL}"   # UPLOAD_TOKEN from the repo\'s .env')
except RuntimeError:
    print(f"Publishing failed (see above). The report is safe at {report}; run this cell again when the API is up.")


## 8. PPO on your laptop (free)
PPO is CPU-only, so it doesn't need Colab. With [Google Drive for desktop](https://www.google.com/drive/download/) syncing the same `runs` folder:

```bash
cd fast-and-slow/backend
uv sync --extra gpu        # the `emb` variant needs the sentence encoder
RUNS="<your Drive folder>/fast-and-slow/runs"
uv run fns ppo-all --profile full --runs "$RUNS"
uv run fns shift --part cpu --profile full --runs "$RUNS"   # H6: retrain the deployed PPO on each new world
```

It needs the note bank from section 3. Like everything else, it resumes after interruptions (every 25k decisions).

## 9. End of session
Run this before disconnecting whenever you stop early (the RL cell does it for you when its time runs out).
Files written to Drive keep uploading after the write returns; deleting the runtime too soon loses them.


In [ ]:
from google.colab import drive
drive.flush_and_unmount()   # waits until everything written to Drive is uploaded
print("Everything is on Drive. Safe to disconnect (Runtime > Disconnect and delete runtime).")
